In [2]:
import pandas as pd
import numpy as np

In [3]:
gen_path = "Split_data/Train/Gen/"
kia_path = "Split_data/Train/Kia/"
sil_path = "Split_data/Train/Sil/"
tes_path = "Split_data/Train/Tesla/"

In [4]:
Gen_AF = pd.read_csv(gen_path + "Gen_AF.csv")
Kia_AF = pd.read_csv(kia_path + "Kia_AF.csv")
Sil_AF = pd.read_csv(sil_path + "Sil_AF.csv")
Tesla_AF = pd.read_csv(tes_path + "Tesla_AF.csv")

/tmp/ipykernel_140705/2111813665.py:1: DtypeWarning: Columns (20,22) have mixed types. Specify dtype option on import or set low_memory=False.
  Gen_AF = pd.read_csv(gen_path + "Gen_AF.csv")


In [6]:
Kia_AF.columns

Index(['Time_Offset', 'CAN_ID', 'Data_Length', 'One', 'Two', 'Three', 'Four',
       'Five', 'Six', 'Seven', 'Eight', 'Label'],
      dtype='object')

In [20]:
import numpy as np

def summarize_can_df(df, name="Genesis"):
    """
    Summarize CAN log statistics from a DataFrame.

    Expects:
        - df['Time_Offset'] in milliseconds (monotonic increasing preferred)
        - df['CAN_ID']
    """
    if "Time_Offset" not in df.columns or "CAN_ID" not in df.columns:
        missing = [c for c in ["Time_Offset", "CAN_ID"] if c not in df.columns]
        raise ValueError(f"Missing required column(s): {missing}")

    # Ensure sorted by time for diff-based gap
    df_sorted = df.sort_values("Time_Offset", kind="mergesort").reset_index(drop=True)

    t0 = df_sorted["Time_Offset"].iloc[0]
    msgs_in_first_sec = (df_sorted["Time_Offset"] <= (t0 + 1000)).sum()

    n_unique_ids = df_sorted["CAN_ID"].nunique()

    inter_frame_gaps = df_sorted["Time_Offset"].diff()
    mean_gap = float(inter_frame_gaps.mean(skipna=True))

    print(f"{name} Total Data Generation Per Second: {msgs_in_first_sec}")
    print(f"{name} Total CAN IDs: {n_unique_ids}")
    print(f"{name} Total Mean Inter-Frame Time Gap: {mean_gap}")

    return {
        "msgs_in_first_sec": int(msgs_in_first_sec),
        "n_unique_ids": int(n_unique_ids),
        "mean_inter_frame_gap": mean_gap,
    }

# Example usage:
# stats = summarize_can_df(Gen_AF, name="Genesis")

In [21]:
stats = summarize_can_df(Gen_AF, name="Genesis")

Genesis Total Data Generation Per Second: 2515
Genesis Total CAN IDs: 58
Genesis Total Mean Inter-Frame Time Gap: 0.3967027845046408


In [22]:
stats = summarize_can_df(Kia_AF, name="Kia Soul")

Kia Soul Total Data Generation Per Second: 2632
Kia Soul Total CAN IDs: 79
Kia Soul Total Mean Inter-Frame Time Gap: 0.38019846699744503


In [23]:
stats = summarize_can_df(Sil_AF, name=" Silverado")

 Silverado Total Data Generation Per Second: 2571
 Silverado Total CAN IDs: 98
 Silverado Total Mean Inter-Frame Time Gap: 0.38897346995578325


In [24]:
stats = summarize_can_df(Tesla_AF, name=" Tesla")

 Tesla Total Data Generation Per Second: 3124
 Tesla Total CAN IDs: 69
 Tesla Total Mean Inter-Frame Time Gap: 0.3193589772649621


In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("/home/lisa/Arupreza/UIDS/UIDS-II/Split_data/OtherLAB")
VALID_EXTS = {".csv"}
AF_SUFFIXES = ("_AF.csv", "_af.csv")

def count_rows_fast_csv(fp: Path) -> int:
    """Count data rows in CSV assuming 1 header row."""
    with fp.open("rb") as f:
        n_lines = 0
        while True:
            chunk = f.read(8 * 1024 * 1024)
            if not chunk:
                break
            n_lines += chunk.count(b"\n")
    return max(0, n_lines - 1)

def is_attack_free_file(fp: Path) -> bool:
    return fp.suffix.lower() in VALID_EXTS and fp.name.endswith(AF_SUFFIXES)

def summarize_otherlab_table(base_dir: Path, source_name: str = "OtherLAB") -> pd.DataFrame:
    """
    Layout:
      base_dir/<Vehicle>/<AttackType>/*.csv
    Each <AttackType> folder also contains <Vehicle>_AF.csv (repeated),
    so we count Attack-Free ONCE per vehicle by de-duplicating AF filenames.
    """
    rows = []
    for vehicle_dir in sorted([p for p in base_dir.iterdir() if p.is_dir()]):
        vehicle = vehicle_dir.name

        # ---- Attack-Free (dedupe repeated AF copies by filename; take max rows for that filename) ----
        af_files = [fp for fp in vehicle_dir.rglob("*.csv") if fp.is_file() and is_attack_free_file(fp)]
        af_by_name = {}
        for fp in af_files:
            af_by_name.setdefault(fp.name, []).append(fp)

        af_total = 0
        for fname, fps in af_by_name.items():
            # repeated across folders -> assume duplicates; take the largest count
            best_cnt = max(count_rows_fast_csv(f) for f in fps)
            af_total += best_cnt

        if af_total > 0:
            rows.append({"Vehicle": vehicle, "Source": source_name, "Type": "Attack-Free", "Total Data": int(af_total)})

        # ---- Attacks: sum non-AF files in each attack folder ----
        for attack_dir in sorted([p for p in vehicle_dir.iterdir() if p.is_dir()]):
            attack_type = attack_dir.name

            total = 0
            for fp in sorted(attack_dir.glob("*.csv")):
                if fp.is_file() and not is_attack_free_file(fp):
                    total += count_rows_fast_csv(fp)

            if total > 0:
                rows.append({"Vehicle": vehicle, "Source": source_name, "Type": attack_type, "Total Data": int(total)})

    df = pd.DataFrame(rows)

    # order like your example
    type_order = ["Attack-Free", "DoS", "Fuzz", "Replay", "Malfunction", "Gear Spoofing", "RPM Spoofing"]
    df["__rank__"] = df["Type"].apply(lambda x: type_order.index(x) if x in type_order else 999)
    df = df.sort_values(["Vehicle", "__rank__", "Type"]).drop(columns="__rank__").reset_index(drop=True)
    return df

# ---- run + show table ----
df = summarize_otherlab_table(BASE_DIR, source_name="OtherLAB")
print(df.to_string(index=False))

# optional: save CSV
out_csv = BASE_DIR.parent / "OtherLAB_summary_table.csv"
df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

 Vehicle   Source          Type  Total Data
Forester OtherLAB   Attack-Free      120000
Forester OtherLAB           DoS       48500
Forester OtherLAB          Fuzz       96000
     Gen OtherLAB   Attack-Free      600000
     Gen OtherLAB           DoS      600000
     Gen OtherLAB          Fuzz      600000
     Kia OtherLAB   Attack-Free      300000
     Kia OtherLAB           DoS       36805
     Kia OtherLAB          Fuzz      342355
     Kia OtherLAB        Replay      363145
     Kia OtherLAB   Malfunction      266445
     Kia OtherLAB Gear Spoofing      500000
     Kia OtherLAB  RPM Spoofing      500000
     Sil OtherLAB   Attack-Free      120000
     Sil OtherLAB           DoS      725000
     Sil OtherLAB          Fuzz      241000
  Sonata OtherLAB   Attack-Free      200000
  Sonata OtherLAB           DoS       36030
  Sonata OtherLAB          Fuzz      272595
  Sonata OtherLAB        Replay      340120
  Sonata OtherLAB   Malfunction      269273

Saved: /home/lisa/Arupreza/UIDS

In [12]:
from utils import SegmentFromFile

ImportError: cannot import name 'SegmentFromFile' from 'utils' (/home/lisa/Arupreza/UIDS/UIDS-II/utils.py)

In [13]:
from utils import SegmentForValidation, SegmentForInference